# 开盘首分钟：任务、因子与 LastPrice IC

当前观察期 **1、2、3、4、5 秒**，默认冷启动 10 秒，每秒产生任务。当前阶段先做单因子 IC；执行策略和模型训练暂缓。主 IC 从观察结束起算，另保留原执行延迟对照。

此入口调用项目共用代码。新原子表复用根目录 `sample_snapshot_原子执行总表.parquet` 文件名，表结构已改为任务、因子与未来标签；不再输出旧 k/H 的执行收益。完整规则见 `撮合规则与研究说明.md`。


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "utils/opening_ic.py").is_file())
sys.path.insert(0, str(ROOT))
if (ROOT / ".cache/python-packages").is_dir():
    sys.path.insert(0, str(ROOT / ".cache/python-packages"))
from scripts.run_opening_ic import run
import pandas as pd
import json


## 参数

`COLD_START_SECONDS` 决定何时开始产生任务；`OBSERVATION_SECONDS` 决定各任务再观察多久。滚动特征历史默认 5、10 秒，未来评价默认 1–5 秒；这些不是同一个窗口。观察和后续标签允许越过首分钟。

运行下面计算单元会重新生成结果；`PUBLISH=True` 表示通过全部检查后替换当前原子表。原始行情保持原样。


In [ ]:
COLD_START_SECONDS = 10.0
OBSERVATION_SECONDS = (1.0, 2.0, 3.0, 4.0, 5.0)
PUBLISH = True


In [ ]:
manifest = run(cold_start=COLD_START_SECONDS, observations=OBSERVATION_SECONDS, publish=PUBLISH)
manifest


## 查看结果

逐日 IC 再对有效日期等权汇总；保留原始正负号。IC 不是正确率或交易收益。全部 40 日此前都曾参与旧项目探索，后 12 日只能称为后续日期验证。完整结果与解释见 `result/opening_execution/IC研究结果.md`。


In [ ]:
OUT = ROOT / "result/opening_execution"
summary = pd.read_csv(OUT / "summary_ic.csv")
view = summary.loc[
    summary.phase.eq("validation")
    & summary.pair_set.eq("common_observations_horizons")
    & summary.target.eq("signed")
    & summary.anchor.eq("decision")
    & summary.observation_seconds.eq(1)
]
view.pivot(index="factor", columns="horizon_seconds", values="mean_rank_ic")
